In [37]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import censusdis.data as ced
from censusdis.datasets import ACS5
from censusdis.states import STATE_VT
import shap

CENSUS_API_KEY = "f74a12ac5b0beb570fc86afe788234103b463fc0"

In [41]:
YEAR = 2022
DATASET = "acs/acs5"
# Median household income: ACS5, Table B19013_001E
try:
    df = ced.download(
        DATASET,
        YEAR,
        ["B19013_001E"],  # median household income in the past 12 months (USD)
        state=STATE_VT,
        zip_code_tabulation_area = "*",
        with_geometry=False
    )
except ced.CensusApiException as e:
    print(e)

# df = df.rename(columns={"B19013_001E": "median_income", "zip code tabulation area": "zcta"})
# df["zcta"] = df["zcta"].astype(str).str.zfill(5)


The following arguments are not recognized as non-geographic arguments or goegraphic arguments
for the dataset acs/acs5 in vintage 2022: 'zip code tabulation area'.

There are two reasons why this might happen:

1. The arg(s) mentioned above are mispelled versions of named or geopgrahic arguments.
2. The arg(s) mentioned above are valid geographic arguments for some data sets and
   vintages, but not for acs/acs5 in vintage 2022.

Supported geographies for dataset='acs/acs5' in year=2022 are:
['us']
['region']
['division']
['state']
['state', 'county']
['state', 'county', 'county_subdivision']
['state', 'county', 'county_subdivision', 'subminor_civil_division']
['state', 'county', 'county_subdivision', 'place_remainder_or_part']
['state', 'county', 'tract']
['state', 'county', 'tract', 'block_group']
['state', 'place', 'county_or_part']
['state', 'place']
['state', 'consolidated_city']
['state', 'consolidated_city', 'place_or_part']
['state', 'alaska_native_regional_corporation']
['am

In [3]:
# Load data
df = pd.read_csv('../data/TTS_LBNL_public_file_29-Sep-2025_all.csv')

/var/folders/sr/hy8nzjhj7xz25lt74m44nv140000gn/T/ipykernel_18141/3527237685.py:2: DtypeWarning: Columns (1,2,3,11,15,16,18,28,29,31,32,34,35,38,39,40,53,54,56,57,59,60,74,75,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/TTS_LBNL_public_file_29-Sep-2025_all.csv')


In [27]:
df['year'] = pd.to_datetime(df['installation_date']).dt.year
df.groupby(['state'])['year'].unique()

state
AR                                   [2010, 2009, 2011]
AZ    [2012, 2010, 2009, 2008, 2006, 2011, 2007, 200...
CA    [2014, 2015, 2016, 2011, 2012, 2013, 2017, 200...
CO    [2017, 2018, 2019, 2020, 2021, 2022, 2009, 201...
CT    [2013, 2010, 2008, 2011, 2007, 2006, 2009, 201...
DC    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
DE    [2008, 2009, 2011, 2014, 2007, 2010, 2012, 201...
FL    [2007, 2006, 2009, 2010, 2008, 2012, 2011, 202...
IL    [1999, 2000, 2001, 2002, 2003, 2004, 2009, 200...
MA    [2003, 2002, 2004, 2005, 2006, 2007, 2008, 201...
MD    [2015, 2017, 2016, 2013, 2014, 2006, 2012, 201...
ME    [2012, 2011, 2013, 1999, 2001, 2004, 2005, 200...
MN    [2015, 2014, 2018, 2016, 2017, 2019, 2007, 200...
NH    [2011, 2012, 2013, 2014, 2015, 2016, 2017, 201...
NJ    [2022, 2021, 2023, 2019, 2020, 2024, 2000, 200...
NM    [2016, 2015, 2014, 2013, 2012, 2011, 2010, 200...
NY    [2023, 2024, 2014, 2012, 2015, 2021, 2010, 202...
OH    [2011, 2008, 2009, 2004, 2007, 2005,

In [19]:
# Subset to california
ca = df[(df['state'] == 'CA') & 
        (df['technology_type'] == 'pv-only') & 
        (df['customer_segment'].isin(["RES", "RES_SF"])) &
        (df['total_installed_price'] > 0) &
        (df['PV_system_size_DC'] > 0)
         ].copy()

# Add a date
ca['year'] = pd.to_datetime(ca['installation_date']).dt.year

# Calculate price per watt figures
ca['price_per_watt'] = ca['total_installed_price']/ca['PV_system_size_DC']/1000

In [ ]:
# Build random forest regressor
cols = ['year', 'PV_system_size_DC', 'self_installed', 'technology_type',
        ]
X = ca[cols]

In [24]:
ca.columns

Index(['data_provider_1', 'data_provider_2', 'system_ID_1', 'system_ID_2',
       'installation_date', 'PV_system_size_DC', 'total_installed_price',
       'rebate_or_grant', 'customer_segment', 'expansion_system',
       'multiple_phase_system', 'TTS_link_ID', 'new_construction', 'tracking',
       'ground_mounted', 'zip_code', 'city', 'state',
       'utility_service_territory', 'third_party_owned', 'installer_name',
       'self_installed', 'azimuth_1', 'azimuth_2', 'azimuth_3', 'tilt_1',
       'tilt_2', 'tilt_3', 'module_manufacturer_1', 'module_model_1',
       'module_quantity_1', 'module_manufacturer_2', 'module_model_2',
       'module_quantity_2', 'module_manufacturer_3', 'module_model_3',
       'module_quantity_3', 'additional_modules', 'technology_module_1',
       'technology_module_2', 'technology_module_3', 'BIPV_module_1',
       'BIPV_module_2', 'BIPV_module_3', 'bifacial_module_1',
       'bifacial_module_2', 'bifacial_module_3', 'nameplate_capacity_module_1',
      

In [22]:
ca.groupby(['year'], as_index = False).agg({'price_per_watt':'median', 'data_provider_1':'count'})

,year,price_per_watt,data_provider_1
0,1998,10.241183,13
1,1999,8.884865,61
2,2000,8.480809,88
3,2001,8.310480,754
4,2002,8.445109,1286
5,2003,7.828956,1861
6,2004,7.473750,3211
7,2005,7.285429,2839
8,2006,7.615057,4683
9,2007,8.029379,7530
